# 03. Procesamiento de datos e ingeniería de variables

## Objetivo
Transformar los datos en bruto en un dataset limpio y unificado, listo para el análisis. El producto final es el **Score de Demanda** por barrio.

## Pasos
1. **Carga de datos**: reimportar los datasets en bruto.
2. **Limpieza y estandarización**: unificar convenios de nombres (códigos de barrio).
3. **Unión espacial**: asignar cada cargador existente a su barrio.
4. **Agregación**: calcular totales de cargadores, renta media y métricas de vehículos por barrio.
5. **Score de Demanda**: crear una métrica compuesta que identifique las zonas con mayor potencial.
6. **Exportación**: guardar el dataset procesado para la siguiente fase.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

search_paths = ['../data/raw', 'data/raw', '../../data/raw']
RAW_DATA_DIR = next((p for p in search_paths if os.path.exists(p)), '../data/raw')
print(f"Cargando datos desde: {os.path.abspath(RAW_DATA_DIR)}")

FILE_GEOMETRY = os.path.join(RAW_DATA_DIR, '0301100100_UNITATS_ADM_POLIGONS.json')
FILE_CHARGERS = os.path.join(RAW_DATA_DIR, '2023_2T_Punts_Recarrega_Vehicle_Electric.json')
FILE_VEHICLES = os.path.join(RAW_DATA_DIR, '2024_parc_vehicles_tipus_propulsio.csv')
FILE_INCOME   = os.path.join(RAW_DATA_DIR, '2022_renda_disponible_llars_per_persona.csv')

Cargando datos desde: /Users/nicolassanvicente/Documents/WIP-EV_hub_optimization copia/data/raw


## 2. Carga de datos

In [2]:
gdf_barris = gpd.read_file(FILE_GEOMETRY)
if gdf_barris.crs != 'EPSG:4326':
    gdf_barris = gdf_barris.to_crs('EPSG:4326')
print(f"Cargadas {len(gdf_barris)} unidades administrativas.")

try:
    gdf_chargers = gpd.read_file(FILE_CHARGERS)
except Exception:
    df_temp = pd.read_json(FILE_CHARGERS)
    gdf_chargers = gpd.GeoDataFrame(
        df_temp,
        geometry=gpd.points_from_xy(df_temp.Station_lng, df_temp.Station_lat),
        crs="EPSG:4326"
    )
print(f"Cargados {len(gdf_chargers)} cargadores.")

df_income = pd.read_csv(FILE_INCOME)
print(f"Renta: {len(df_income)} filas.")

df_vehicles = pd.read_csv(FILE_VEHICLES)
print(f"Vehículos: {len(df_vehicles)} filas.")

Cargadas 1501 unidades administrativas.
Cargados 393 cargadores.
Renta: 1068 filas.
Vehículos: 5787 filas.


## 3. Limpieza y estandarización
Filtramos la geometría para quedarnos únicamente con los polígonos de barrio (`BARRI`) y unificamos el código identificador.

In [3]:
if 'TIPUS_UA' in gdf_barris.columns:
    gdf_barris = gdf_barris[gdf_barris['TIPUS_UA'] == 'BARRI'].copy()
    print(f"Geometría filtrada: {len(gdf_barris)} barrios.")

col_barri_geo = 'BARRI' if 'BARRI' in gdf_barris.columns else next(
    (c for c in gdf_barris.columns if 'BARRI' in c.upper()), 'CODI_UA'
)
print(f"Columna de código de barrio: '{col_barri_geo}'")

gdf_barris['Barri_ID'] = gdf_barris[col_barri_geo].astype(str).str.strip().astype(int)
df_income['Barri_ID'] = df_income['Codi_Barri'].astype(int)

df_vehicles['Barri_ID'] = pd.to_numeric(df_vehicles['Codi_Barri'], errors='coerce')
n_invalid = df_vehicles['Barri_ID'].isna().sum()
print(f"Eliminando {n_invalid} filas con código de barrio no numérico.")
df_vehicles = df_vehicles.dropna(subset=['Barri_ID'])
df_vehicles['Barri_ID'] = df_vehicles['Barri_ID'].astype(int)

print("Estandarización completada. Clave común: 'Barri_ID'.")

Geometría filtrada: 73 barrios.
Columna de código de barrio: 'BARRI'
Eliminando 3 filas con código de barrio no numérico.
Estandarización completada. Clave común: 'Barri_ID'.


## 4. Agregación de datos socioeconómicos

### 4.1 Renta media por barrio

In [4]:
df_income_agg = (
    df_income.groupby('Barri_ID')['Import_Euros']
    .mean()
    .reset_index()
    .rename(columns={'Import_Euros': 'Avg_Income'})
)
display(df_income_agg.head())

,Barri_ID,Avg_Income
0,1,14221.142857
1,2,18668.111111
2,3,17965.909091
3,4,19655.153846
4,5,24163.200000


### 4.2 Vehículos por barrio
Calculamos el total de vehículos y el número de vehículos eléctricos e híbridos.

In [5]:
df_vehicles_agg = (
    df_vehicles.groupby('Barri_ID')['Nombre']
    .sum()
    .reset_index()
    .rename(columns={'Nombre': 'Total_Vehicles'})
)

ev_types = ['Elèctrica', 'Híbrid']
df_ev_agg = (
    df_vehicles[df_vehicles['Tipus_Propulsio'].isin(ev_types)]
    .groupby('Barri_ID')['Nombre']
    .sum()
    .reset_index()
    .rename(columns={'Nombre': 'EV_Count'})
)

df_vehicles_agg = df_vehicles_agg.merge(df_ev_agg, on='Barri_ID', how='left')
df_vehicles_agg['EV_Count'] = df_vehicles_agg['EV_Count'].fillna(0)
display(df_vehicles_agg.head())

,Barri_ID,Total_Vehicles,EV_Count
0,1,11725,897
1,2,10722,525
2,3,9260,2378
3,4,13947,4620
4,5,14321,1206


## 5. Unión espacial: cargadores → barrios
Contamos cuántos cargadores existentes hay dentro de cada polígono de barrio.

In [6]:
gdf_joined = gpd.sjoin(gdf_chargers, gdf_barris, how="left", predicate="within")
chargers_per_barri = gdf_joined.groupby('Barri_ID').size().reset_index(name='Charger_Count')
display(chargers_per_barri.head())

,Barri_ID,Charger_Count
0,1.0,13
1,2.0,10
2,3.0,17
3,4.0,2
4,5.0,67


## 6. Dataset maestro (barrios + demanda)
Consolidamos toda la información en un único GeoDataFrame.

In [7]:
master_df = gdf_barris[['Barri_ID', 'NOM', 'geometry']].copy()
master_df = master_df.merge(df_income_agg, on='Barri_ID', how='left')
master_df = master_df.merge(df_vehicles_agg, on='Barri_ID', how='left')
master_df = master_df.merge(chargers_per_barri, on='Barri_ID', how='left')

master_df['Charger_Count'] = master_df['Charger_Count'].fillna(0)
master_df['Avg_Income'] = master_df['Avg_Income'].fillna(master_df['Avg_Income'].median())
master_df['Total_Vehicles'] = master_df['Total_Vehicles'].fillna(master_df['Total_Vehicles'].median())
master_df['EV_Count'] = master_df['EV_Count'].fillna(0)

display(master_df.head())

,Barri_ID,NOM,geometry,Avg_Income,Total_Vehicles,EV_Count,Charger_Count
0,1,el Raval,"POLYGON ((2.16471 41.38593, 2.16401 41.3854, 2...",14221.142857,11725,897,13.0
1,2,el Barri Gòtic,"POLYGON ((2.17701 41.38525, 2.17658 41.38558, ...",18668.111111,10722,525,10.0
2,3,la Barceloneta,"POLYGON ((2.19623 41.38745, 2.19617 41.38746, ...",17965.909091,9260,2378,17.0
3,4,"Sant Pere, Santa Caterina i la Ribera","POLYGON ((2.18345 41.39061, 2.18238 41.39142, ...",19655.153846,13947,4620,2.0
4,5,el Fort Pienc,"POLYGON ((2.18353 41.39227, 2.18388 41.39253, ...",24163.200000,14321,1206,67.0


## 7. Score de Demanda
Métrica compuesta que pondera tres factores:

- **Vehículos eléctricos** (50 %): indicador directo de necesidad actual de recarga.
- **Renta familiar** (30 %): capacidad de adquisición futura de VE.
- **Total de vehículos** (20 %): densidad de tráfico y tamaño potencial del mercado.

$$\text{Score} = (\text{Norm\_EVs} \times 0{,}5) + (\text{Norm\_Income} \times 0{,}3) + (\text{Norm\_TotalVehicles} \times 0{,}2)$$

In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
master_df['Norm_Income'] = scaler.fit_transform(master_df[['Avg_Income']])
master_df['Norm_Vehicles'] = scaler.fit_transform(master_df[['Total_Vehicles']])
master_df['Norm_EVs'] = scaler.fit_transform(master_df[['EV_Count']])

master_df['Demand_Score'] = (
    master_df['Norm_EVs'] * 0.5
    + master_df['Norm_Income'] * 0.3
    + master_df['Norm_Vehicles'] * 0.2
) * 100

display(
    master_df[['NOM', 'Demand_Score', 'EV_Count', 'Total_Vehicles', 'Avg_Income']]
    .sort_values('Demand_Score', ascending=False)
    .head(10)
)

,NOM,Demand_Score,EV_Count,Total_Vehicles,Avg_Income
25,Sant Gervasi - Galvany,83.468167,3575,36060,36283.193548
18,les Corts,70.953194,3532,27575,29115.228571
6,la Dreta de l'Eixample,68.997667,3181,29640,29815.344828
3,"Sant Pere, Santa Caterina i la Ribera",65.551732,4620,13947,19655.153846
8,la Nova Esquerra de l'Eixample,59.018276,2647,28673,26255.950000
24,Sant Gervasi - la Bonanova,55.413504,1944,18094,36110.222222
11,la Marina del Prat Vermell,55.115824,4020,13594,16015.000000
7,l'Antiga Esquerra de l'Eixample,51.042672,2048,22448,28278.500000
22,Sarrià,49.332645,1517,16757,35446.375000
23,les Tres Torres,48.995498,1147,12680,41344.181818


## 8. Exportación
Guardamos el dataset procesado como GeoJSON para la siguiente fase.

In [9]:
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUT_FILE = os.path.join(PROCESSED_DIR, 'barrios_with_demand.geojson')
master_df.to_file(OUT_FILE, driver='GeoJSON')
print(f"Dataset guardado en: {os.path.abspath(OUT_FILE)}")

Dataset guardado en: /Users/nicolassanvicente/Documents/WIP-EV_hub_optimization copia/data/processed/barrios_with_demand.geojson
